---
format:
  html:
    code-fold: true
jupyter: python3
---

For this assignment, I selected the Breast Cancer Wisconsin (Diagnostic) dataset from Kaggle. It contains 569 samples and 30 continuous features computed from digitized images of fine needle aspirates of breast masses. The classification task is to predict whether a tumor is malignant or benign. This is an excellent problem to solve because it mirrors real-world diagnostic challenges where high-dimensional biological data often contains significant redundancy. Analyzing this dataset provides a strong foundation for exploring structural sparsity and feature pruning. By identifying and eliminating non-essential features without losing predictive power, we can build a more efficient pipeline, effectively separating the diagnostic signal from the noise.

In [1]:
import kagglehub
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Download the dataset programmatically
print("Downloading dataset via kagglehub...")
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")

# 2. Find the CSV file in the downloaded directory
# (Kagglehub saves it in a cache folder, so we just grab the first CSV we see)
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)

# 3. Load into Pandas
df = pd.read_csv(full_path)

# 4. Preprocessing: drop ID and empty Kaggle artifact column if they exist
if 'id' in df.columns: 
    df = df.drop(columns=['id'])
if 'Unnamed: 32' in df.columns: 
    df = df.drop(columns=['Unnamed: 32'])

# Encode the categorical target ('M' = Malignant, 'B' = Benign)
le = LabelEncoder()
y = le.fit_transform(df['diagnosis'])
X = df.drop(columns=['diagnosis'])

# Standard Scaling 
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 80/20 Train-Test Split 
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.20, random_state=42)
print(f"Training features shape: {X_train.shape}, Testing features shape: {X_test.shape}")

/home/min/a/pmaletti/miniconda3/envs/amoaballm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 48.6k/48.6k [00:00<00:00, 1.07MB/s]

Extracting files...
Training features shape: (455, 30), Testing features shape: (114, 30)


To optimize both baseline models, a systematic grid search with 5-fold cross-validation will be used. For k-Nearest Neighbors, I will test k in [3, 5, 7, 9, 11, 13, 15]. Using odd numbers prevents tie-breaking issues in binary classification, and this range strikes a balance between capturing local data geometries and smoothing out outlier noise. For Logistic Regression, I will tune the inverse regularization strength, C, across a logarithmic scale [0.01, 0.1, 1, 10, 100]. This range allows the model to explore extreme parameter penalties (preventing overfitting in our 30-feature space) up to nearly unregularized states, ensuring we identify the optimal bias-variance tradeoff.

In [2]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# KNN Hyperparameter Tuning
knn_params = {'n_neighbors': [3, 5, 7, 9, 11, 13, 15]}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5)
knn_grid.fit(X_train, y_train)
knn_best = knn_grid.best_estimator_

# Logistic Regression Hyperparameter Tuning
lr_params = {'C': [0.01, 0.1, 1, 10, 100]}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000), lr_params, cv=5)
lr_grid.fit(X_train, y_train)
lr_best = lr_grid.best_estimator_

# Evaluation on held-out test set
knn_acc = accuracy_score(y_test, knn_best.predict(X_test))
lr_acc = accuracy_score(y_test, lr_best.predict(X_test))

print(f"Best KNN (k={knn_grid.best_params_['n_neighbors']}) Test Accuracy: {knn_acc:.4f}")
print(f"Best Logistic Regression (C={lr_grid.best_params_['C']}) Test Accuracy: {lr_acc:.4f}")

Best KNN (k=5) Test Accuracy: 0.9474
Best Logistic Regression (C=1) Test Accuracy: 0.9737


Biological diagnostic datasets often exhibit high feature redundancy (e.g., cell radius, perimeter, and area are highly correlated). I hypothesize that applying ANOVA F-value-based feature selection will allow us to enforce structural sparsity and confidently drop exactly 50% of the features (15 out of 30). I predict this pruning strategy will marginally improve, or at least competitively match, the baseline performance. By dropping noisy and redundant features, we effectively simplify the decision boundary and reduce the curse of dimensionality for KNN, creating a leaner classification pipeline without sacrificing the core diagnostic signals.

In [3]:
from sklearn.feature_selection import SelectKBest, f_classif

# Enforce 50% pruning (drop 15 of 30 features)
k_best = X_train.shape[1] // 2
pruner = SelectKBest(score_func=f_classif, k=k_best)

X_train_pruned = pruner.fit_transform(X_train, y_train)
X_test_pruned = pruner.transform(X_test)

print(f"Original feature count: {X_train.shape[1]} -> Pruned feature count: {X_train_pruned.shape[1]}\n")

# Re-run Grid Search pipelines on the reduced dataset
knn_grid.fit(X_train_pruned, y_train)
lr_grid.fit(X_train_pruned, y_train)

knn_pruned_acc = accuracy_score(y_test, knn_grid.best_estimator_.predict(X_test_pruned))
lr_pruned_acc = accuracy_score(y_test, lr_grid.best_estimator_.predict(X_test_pruned))

print(f"Pruned KNN Test Accuracy: {knn_pruned_acc:.4f}")
print(f"Pruned Logistic Regression Test Accuracy: {lr_pruned_acc:.4f}")
print("\nComment: Pruning 50% of the features successfully maintained strong accuracy, confirming that significant correlation existed and the sparsified dataset retained the essential predictive information.")

Original feature count: 30 -> Pruned feature count: 15

Pruned KNN Test Accuracy: 0.9649
Pruned Logistic Regression Test Accuracy: 0.9649

Comment: Pruning 50% of the features successfully maintained strong accuracy, confirming that significant correlation existed and the sparsified dataset retained the essential predictive information.


To surpass the baseline benchmarks, I propose building a PyTorch-based Multilayer Perceptron (MLP) classifier. While Logistic Regression is limited to linear decision boundaries, a neural network utilizing non-linear ReLU activations can capture complex, higher-order interactions among the pruned biological features. Given the reduced dimensionality of our sparsified dataset, a compact feedforward network optimized with Adam should train efficiently without overfitting, allowing it to outperform standard linear and distance-based classifiers.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# Convert pruned NumPy arrays to PyTorch tensors
X_tr_tensor = torch.tensor(X_train_pruned, dtype=torch.float32)
y_tr_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_te_tensor = torch.tensor(X_test_pruned, dtype=torch.float32)
y_te_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Define a compact Neural Network
class DiagnosticMLP(nn.Module):
    def __init__(self, input_dim):
        super(DiagnosticMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.network(x)

# Initialize model, loss function, and Adam optimizer
model = DiagnosticMLP(X_train_pruned.shape[1])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    predictions = model(X_tr_tensor)
    loss = criterion(predictions, y_tr_tensor)
    loss.backward()
    optimizer.step()

# Evaluation on test set
model.eval()
with torch.no_grad():
    test_preds = (model(X_te_tensor) >= 0.5).float()
    pytorch_acc = (test_preds == y_te_tensor).float().mean().item()

print(f"PyTorch MLP Test Accuracy on Pruned Data: {pytorch_acc:.4f}")

PyTorch MLP Test Accuracy on Pruned Data: 0.9825
